# Ground-truth vs. LLM answer evaluation
This notebook evaluates answers you have already generated; it does not generate your model's answers.

## Recommended method
Use an LLM-as-judge rubric for the main success score, with embedding cosine similarity as a supporting diagnostic. Two correct answers may use different wording, so cosine similarity alone is not a reliable correctness metric.

The judge evaluates semantic agreement, coverage of important points, contradictions, and unsupported claims. It returns a score from 0-5, converted to 0-100. You can set your own pass threshold and inspect each item's rationale.

In [ ]:
!pip install --no-cache-dir \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    torchvision==0.26.0 \
    --index-url https://download.pytorch.org/whl/cu130

!pip install -U transformers accelerate

!pip install -q pandas numpy scikit-learn sentence-transformers bottleneck


Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 119.7 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 243.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 256.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 204.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.1/531.1 MB 164.5 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 171.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 53.9 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 122.7 MB/s  0:00:01
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.1
    Uninstalling nvidia-cusparselt-cu13-0.8.1:
      Successfully uninstalled nvidia-cusparselt-cu13-0.8.1
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.3.0


In [ ]:
!pip install -U transformers accelerate huggingface_hub safetensors

from huggingface_hub import snapshot_download
import os

model_id = "Qwen/Qwen3.5-4B"
model_path = "/content/drive/MyDrive/models/Qwen3.5-4B"

# Create destination directory
os.makedirs(model_path, exist_ok=True)

# Download the complete model
snapshot_download(
    repo_id=model_id,
    local_dir=model_path
)

print(f"Model successfully downloaded to: {model_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 97.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 37.4 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have 

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Model successfully downloaded to: /content/drive/MyDrive/models/Qwen3.5-4B


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Evaluation data file. Question, ground truth, category, etc.
TESTING_DATA_PATH = "test_data.json"

# Outputs by the LLM
RAG_OUTPUTS_PATH = None

def load_json_records(path: str) -> json:
  """Load JSON records from a provided path"""
  path = Path(path)
  content = path.read_text(encoding="utf-8")
  return json.loads(content)

def combine_data(test_data_path: str, rag_outputs_path: str) -> pd.DataFrame:
  test_data = load_json_records(path=test_data_path)
  rag_output_data = load_json_records(path=rag_outputs_path)

  # Create ground-truth map for concatenating to the RAG outputs dict
  response_map = {}

  for rag_output_entry in rag_output_data:
    response_map[rag_output_data["question"]] = rag_output_data["ground_truth"]

  # Combine datasets
  combined_data = []

  # Keep within RAGAS format
  for output in test_data:
    combined_item = {
      "id": output["id"],
      "type": output["type"],
      "domains": output["domains"],
      "primary_domain": output["primary_domain"],
      "user_input": output["question"],
      "retrieved_contexts": output["retrieved_contexts"],
      # Output from the RAG system
      "response": response_map.get(output["ground_truth"], ""),
      "reference": output["ground_truth"],
      "is_unanswerable": output["is_unanswerable"]
    }

    combined_data.append(combined_item)
  combined_df = pd.DataFrame(combined_data)
  combined_df[["id", "type", "domains", "primary_domain", "user_input", "response", "reference", "is_unanswerable"]].head()

  return combined_df

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def compute_cosine_similarity(combined_df: pd.DataFrame):

  EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"

  embedder = SentenceTransformer(EMBEDDING_MODEL)
  refs = combined_df["ground_truth"].fillna("").astype(str).tolist()
  preds = combined_df["response"].fillna("").astype(str).tolist()
  ref_vecs = embedder.encode(refs, normalize_embeddings=True, show_progress_bar=True)
  pred_vecs = embedder.encode(preds, normalize_embeddings=True, show_progress_bar=True)
  combined_df["embedding_similarity"] = (ref_vecs * pred_vecs).sum(axis=1)
  combined_df[["id", "embedding_similarity"]].head()

In [ ]:
!pip install -U huggingface_hub

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextIteratorStreamer,
)
import threading
import torch

model_id = "Qwen/Qwen3.5-4B"
model_path = "/content/drive/MyDrive/models/Qwen3.5-4B"


def load_model(model_id=model_id, model_path=model_path):
    """Load Qwen3.5-4B."""

    if not torch.cuda.is_available():
        raise SystemExit("No CUDA GPU found.")

    print(f"GPU: {torch.cuda.get_device_name(0)}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=dtype,
        device_map="auto",
        local_files_only=True,
        trust_remote_code=True,
    )

    model.eval()

    return tokenizer, model

In [ ]:
import threading
import warnings
import contextlib
import os
from transformers import TextIteratorStreamer


def generate(
    tokenizer,
    model,
    messages,
    user_temperature=0.5,
    think=True,
    max_new_tokens=64000,
    verbose=False,
):
    """Generate response incrementally using token streaming."""

    # Suppress Python warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=think,
        )

        inputs = tokenizer(
            [input_text],
            return_tensors="pt",
        ).to(model.device)

    # Sampling settings
    if think:
        sampling = dict(
            temperature=max(user_temperature - 0.1, 0.01),
            top_p=0.95,
            top_k=20,
        )
    else:
        sampling = dict(
            temperature=user_temperature,
            top_p=0.8,
            top_k=20,
        )

    # Streamer
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=False,
        timeout=120,
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        **sampling,
    )

    # Generate in background thread
    def generate_silently():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            # Suppress stderr warnings
            with open(os.devnull, "w") as devnull:
                with contextlib.redirect_stderr(devnull):
                    model.generate(**generation_kwargs)

    thread = threading.Thread(target=generate_silently)
    thread.start()

    thinking = ""
    answer = ""
    in_thinking = think

    if verbose:
        print("\nGenerating...\n")

    # Consume generated text incrementally
    for text in streamer:

        if not text:
            continue

        # Detect end of thinking
        if in_thinking and "</think>" in text:

            before, after = text.split("</think>", 1)

            thinking += before

            in_thinking = False

            if verbose:
                print("\n\n--- Answer ---\n", end="", flush=True)

            answer += after

            if verbose:
                print(after, end="", flush=True)

        elif in_thinking:

            thinking += text

            if verbose:
                print(text, end="", flush=True)

        else:

            answer += text

            if verbose:
                print(text, end="", flush=True)

    thread.join()

    if verbose:
        print("\n\nGeneration complete.")

    return thinking.strip(), answer.strip()

In [ ]:
def ask(
    tokenizer,
    model,
    system_prompt: str,
    question: str,
    think=False,
    max_new_tokens=64000,
    user_temperature=0.5,
    verbose=True
):
    """Send question and stream model output."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    thinking, answer = generate(
        tokenizer=tokenizer,
        model=model,
        messages=messages,
        user_temperature=user_temperature,
        think=think,
        max_new_tokens=max_new_tokens,
        verbose=verbose,
    )

    return thinking, answer

In [ ]:
import time
import re

JUDGE_SYSTEM_PROMPT = '''
You are a strict but fair evaluator comparing a candidate answer against a
ground-truth reference. Evaluate meaning, not wording: accept paraphrases,
equivalent explanations, and concise answers.

Score each dimension with an integer from 0 to 5:

- correctness: Information is factually consistent with the ground truth.
- coverage: Essential points, conditions, and recommendations are included.
- consistency: No contradictions of the ground truth (5 = none, 0 = pervasive).
- groundedness: No unsupported, speculative, or fabricated claims (5 = none).
- relevance: The answer addresses the actual question.

Scale: 5 = fully satisfied; 4 = minor gaps; 3 = partially satisfied;
2 = major gaps; 1 = barely satisfied; 0 = not satisfied.

Rules:
- Weigh omissions and errors by importance.
- Use "primary_domain" first; for multi-domain questions, also assess
  coverage of the other listed "domains" as emphasized in the ground truth.
- If "is_unanswerable" is true, a clear acknowledgement of insufficient
  information is correct; penalize fabricated or speculative personal
  information under correctness and groundedness.
- Do not reward keyword matching without semantic correctness.
- Provide "reasons" as a list of concise strings explaining the scores.
  Include one string for each dimension, identifying important errors,
  omissions, contradictions, unsupported claims, or strengths as applicable.

Return ONLY valid JSON (double quotes, no trailing commas, no extra text):

{
  "correctness": 0,
  "coverage": 0,
  "consistency": 0,
  "groundedness": 0,
  "relevance": 0,
  "reasons": [
    "Correctness: ...",
    "Coverage: ...",
    "Consistency: ...",
    "Groundedness: ...",
    "Relevance: ..."
  ]
}

EXAMPLES

QUESTION:
"I am an international student who has started a paid internship.
My insurer says my contributions may change. Why can employment
affect my insurance?"

GROUND TRUTH:
"Employment type, hours, remuneration and whether an internship
is mandatory or voluntary can affect social-insurance treatment.
Student status alone does not settle every case. Give the insurer
and employer accurate study and employment details and request
a written classification before estimating take-home pay."

Example 1 — Fully correct answer

CANDIDATE ANSWER:
"Your insurance contributions can depend on the type of employment,
working hours, and salary. Mandatory and voluntary internships may
also be treated differently. Being a student does not automatically
determine your insurance status, so you should provide your insurer
and employer with accurate details and request written confirmation
of your classification."

YOUR EXPECTED OUTPUT:
{
  "correctness": 5,
  "coverage": 5,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 5,
  "reasons": [
    "Correctness: The explanation aligns with the reference.",
    "Coverage: It includes the key factors and recommended actions.",
    "Consistency: It does not contradict the reference.",
    "Groundedness: It makes no unsupported claims.",
    "Relevance: It directly answers the question."
  ]
}

Example 2 — Partially correct answer

CANDIDATE ANSWER:
"Your insurance contributions may change because you are earning
money through employment. You should contact your insurer to
check whether your contributions will increase."

YOUR EXPECTED OUTPUT:
{
  "correctness": 3,
  "coverage": 1,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 3,
  "reasons": [
    "Correctness: It gives a plausible but incomplete explanation.",
    "Coverage: It omits most classification factors and recommended steps.",
    "Consistency: It does not contradict the reference.",
    "Groundedness: Its claims are supported by the reference.",
    "Relevance: It addresses the question but gives limited explanation."
  ]
}

Example 3 — Incorrect answer

CANDIDATE ANSWER:
"Your insurance contributions cannot change because you are a
student. All international students are automatically exempt
from social-insurance contributions, regardless of their salary,
working hours, or type of internship."

YOUR EXPECTED OUTPUT:
{
  "correctness": 0,
  "coverage": 0,
  "consistency": 0,
  "groundedness": 0,
  "relevance": 1,
  "reasons": [
    "Correctness: The claimed automatic exemption is false.",
    "Coverage: It omits the essential explanation and recommendations.",
    "Consistency: It directly contradicts the reference.",
    "Groundedness: Its blanket exemption claim is unsupported.",
    "Relevance: It addresses the topic but fails to answer correctly."
  ]
}
'''.strip()

METRICS = (
  "correctness",
  "coverage",
  "consistency",
  "groundedness",
  "relevance",
)


class EvaluationPipeline:
  """Automated evaluation of a given dataframe or single instance using a judge LLM"""
  def __init__(self, eval_df: pd.DataFrame=pd.DataFrame(), metrics: list=METRICS, system_prompt: str=JUDGE_SYSTEM_PROMPT):
      # Load model ONCE
      self.tokenizer, self.model = load_model()
      self.system_prompt = system_prompt
      self.evaluation_dataframe = eval_df
      self.eval_metrics = metrics

  def make_compare_prompts(self, df_row: pd.Series):
    """From dataframe series: Generate prompt for comparison between the candidate answer and the ground truth"""
    metadata = {
        "id": str(df_row.get("id", "")),
        "type": df_row.get("type"),
        "domains": df_row.get("domains"),
        "primary_domain": df_row.get("primary_domain"),
        "is_unanswerable": df_row.get("is_unanswerable", False),
    }
    return f'''
      Metadata:
      {json.dumps(metadata, ensure_ascii=False, indent=2)}

      USER INPUT:
      {df_row.get("user_input", "")}

      GROUND TRUTH:
      {df_row["reference"]}

      CANDIDATE LLM ANSWER:
      {df_row["response"]}

      Compare the candidate with the ground truth using the rubric. Return only the required JSON.
    '''.strip()


  def make_compare_prompts(self, user_input: str, ground_truth: str, candidate_answer: str, metadata: dict):
    """From strings: Generate prompt for comparison between the candidate answer and the ground truth"""
    return f'''
      Metadata:
      {json.dumps(metadata, ensure_ascii=False, indent=2)}

      USER INPUT:
      {user_input}

      GROUND TRUTH:
      {ground_truth}

      CANDIDATE LLM ANSWER:
      {candidate_answer}

      Compare the candidate with the ground truth using the rubric. Return only the required JSON.
    '''.strip()


  def parse_judge_json(self, raw: str):
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", raw, flags=re.I)
    result = json.loads(raw)

    if not isinstance(result, dict):
      raise ValueError("Judge response must be a JSON object")

    # Check for metric within proper range
    for metric in self.eval_metrics:
      score = int(result.get(metric))
      if (0 <= score <= 5):
        raise ValueError(
            f"Judge {metric} score must be a number from 0 to 5; "
            f"got {score!r}")
    return result


  def judge_candidate_answer(self, df_row: pd.Series) -> dict:
    """From dataframe series: Generate jduge's answer to the comparison of candidate answer and ground truth"""
    # Construct judge query from the dataframe row
    judge_query = self.make_compare_prompts(df_row=df_row)

    raw_judge_verdict = ask(
      tokenizer=self.tokenizer,
      model=self.model,
      system_prompt=self.system_prompt,
      question=judge_query,
      think=False,
      verbose=False
    )

    if !verbose:
      # Parse raw LLM output into JSON object
      result = self.parse_judge_json(raw_judge_verdict)
    else:
      result = raw_judge_verdict

    return result


  def judge_candidate_answer(self, user_input: str,
                             ground_truth: str,
                             candidate_answer: str,
                             metadata: dict,
                             verbose=True) -> dict:
    """From strings: Generate jduge's answer to the comparison of candidate answer and ground truth"""
    # Construct judge query from the dataframe row
    judge_query = self.make_compare_prompts(user_input=user_input, ground_truth=ground_truth, candidate_answer=candidate_answer, metadata=metadata)

    raw_judge_verdict = ask(
      tokenizer=self.tokenizer,
      model=self.model,
      system_prompt=self.system_prompt,
      question=judge_query,
      think=False,
      verbose=verbose,
    )
    if !verbose:
      # Parse raw LLM output into JSON object
      result = self.parse_judge_json(raw_judge_verdict)
    else:
        result = raw_judge_verdict

    return result

In [ ]:
eval_pipeline = EvaluationPipeline()
USER_INPUT = "What is Nebenkosten?"
GROUND_TRUTH = "Nebenkosten are additional operating costs associated with a property, such as certain water, waste, cleaning or building-service costs. The lease and annual statement determine the applicable items."
CANDIDATE_ANSWER = """
**Nebenkosten** are colloquially referred to as **Betriebskosten** (operating costs). These are costs that arise in addition to the rent payment.

According to the Betriebskostenverordnung (BetrkV), which applies to both owner-occupied and rented properties:
*   **§ 1 BetrKV** defines what costs cannot be passed on to tenants.
*   **§ 2 BetrKV** specifies which items can be passed on (umgelegt).

Common examples of passable operating costs include water, heating, personal costs, maintenance, cleaning, and insurance. Costs directly resulting from the tenant's own consumption (like water and heating) are typically calculated based on usage, while other costs are often distributed using a key based on apartment size or the number of tenants
"""
METADATA = {
    "id": "S32",
    "type": "simple_factual",
    "domains": [
      "Rent and moving"
    ],
    "primary_domain": "Rent and moving",
    "is_unanswerable": False
}
eval_pipeline.judge_candidate_answer(user_input=USER_INPUT, ground_truth=GROUND_TRUTH,
                                     candidate_answer=CANDIDATE_ANSWER, metadata=METADATA, verbose=False)
print(eval_pipeline)


GPU: Tesla T4


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]


Generating...



[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{
  "correctness": 4,
  "coverage": 3,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 5,
  "reasons": [
    "Correctness: The definition of Nebenkosten as additional operating costs is accurate and consistent with the ground truth.",
    "Coverage: The answer omits the specific mention of the lease and annual statement as the determinants for applicable items, which is a key part of the ground truth.",
    "Consistency: The candidate answer does not contradict the ground truth and provides valid additional context.",
    "Groundedness: All claims made by the candidate are factually correct and supported by general knowledge regarding German tenancy law.",
    "Relevance: The answer directly addresses the question of what Nebenkosten are."
  ]
}<|im_end|>
<|endoftext|>

Generation complete.


AttributeError: 'tuple' object has no attribute 'strip'

In [ ]:
USER_INPUT = "What is a Kaution?"
GROUND_TRUTH = "A Kaution is a rental security deposit intended to secure certain landlord claims under the tenancy. For residential leases, the statutory maximum is generally three months' net cold rent, and tenants have a statutory right to pay it in three monthly instalments."
CANDIDATE_ANSWER = """
Based on the provided context, a **Kaution** (security deposit) is a sum of money that tenants must deposit as security when concluding a rental contract. It is also referred to as **Mietsicherheit** (rental security).

Key characteristics include:
*   **Purpose:** It serves as security for the rental relationship.
*   **Legal Status:** It is not legally mandatory to agree on a Kaution, though it is standard practice in both residential and commercial tenancy law.
*   **Usage:** Tenants cannot use the Kaution to pay rent during the notice period.
*   **Return:** It is typically refunded if there are no outstanding payments and no damages recorded in the handover protocol.
*   **Retention:** Landlords may retain the Kaution even after the statute of limitations expires, provided they have a valid reason (e.g., damages), though they must be able to substantiate their claims.
"""
METADATA = {
    "id": "S32",
    "type": "simple_factual",
    "domains": [
      "Rent and moving"
    ],
    "primary_domain": "Rent and moving",
    "is_unanswerable": False
}
eval_pipeline.judge_candidate_answer(user_input=USER_INPUT, ground_truth=GROUND_TRUTH,
                                     candidate_answer=CANDIDATE_ANSWER, metadata=METADATA, verbose=False)
print(eval_pipeline)


Generating...

{
  "correctness": 4,
  "coverage": 2,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 5,
  "reasons": [
    "Correctness: The definition and general purpose of a Kaution are accurate, but the answer contradicts the ground truth by stating it is not legally mandatory, whereas the ground truth implies statutory limits exist for it.",
    "Coverage: The answer omits specific details provided in the ground truth, such as the statutory maximum amount (three months' net cold rent) and the right to pay in three monthly instalments.",
    "Consistency: The answer does not contradict the ground truth on the core definition, though it diverges on the mandatory nature and misses specific statutory details.",
    "Groundedness: All claims made by the candidate are factually correct in the context of German tenancy law and do not fabricate information.",
    "Relevance: The answer directly addresses the user's question about what a Kaution is."
  ]
}<|im_end|>
<|endoftext|>

AttributeError: 'tuple' object has no attribute 'strip'

In [ ]:
#-------------------------SQL CONFIG----------------------#

EVALUATION_SCHEMA = """
-- =====================================================
-- TABLE 1: EVALUATION TABLE
-- Represents one complete evaluation run
-- =====================================================

CREATE TABLE IF NOT EXISTS evaluation_table (
    evaluation_table_id     TEXT PRIMARY KEY,

    -- Model configuration
    generative_model_id     TEXT NOT NULL,
    embedding_model_id      TEXT,
    evaluator_model_id      TEXT NOT NULL,

    -- RAG configuration
    rag_type                TEXT NOT NULL,

    -- Evaluation configuration
    evaluation_version      TEXT,
    evaluation_config       TEXT,

    -- Dataset configuration
    dataset_id              TEXT,
    dataset_version         TEXT,
);


-- =====================================================
-- TABLE 2: EVALUATION ENTRY
-- Represents one evaluated test case
-- =====================================================

CREATE TABLE IF NOT EXISTS evaluation_entry (
    evaluation_id           INTEGER PRIMARY KEY AUTOINCREMENT,

    -- Relationship to evaluation_table
    evaluation_table_id     TEXT NOT NULL,

    -- Dataset entry reference
    entry_id                TEXT NOT NULL,

    -- Input and reference data
    user_input              TEXT,
    ground_truth            TEXT,

    -- RAG output
    retrieved_context       TEXT,

    -- Generated answer
    candidate_answer        TEXT,

    -- Semantic evaluation
    semantic_score          REAL,

    -- Individual judge scores (e.g. 1-5)
    correctness              INTEGER DEFAULT 3,
    coverage                 INTEGER DEFAULT 3,
    groundedness             INTEGER DEFAULT 3,
    relevance                INTEGER DEFAULT 3,

    -- Overall evaluation
    score                    REAL,
    reasons                  TEXT,

    -- Evaluation result
    success                  INTEGER DEFAULT 0
        CHECK (success IN (0, 1)),

    -- Additional information
    metadata                 TEXT,

    -- Foreign key constraint
    FOREIGN KEY (evaluation_table_id)
        REFERENCES evaluation_table(evaluation_table_id)
        ON DELETE CASCADE,

    -- Prevent duplicate test cases within the same run
    UNIQUE (evaluation_table_id, entry_id)
);


-- =====================================================
-- INDEXES
-- =====================================================

CREATE INDEX IF NOT EXISTS idx_evaluation_entry_table_id
ON evaluation_entry(evaluation_table_id);

CREATE INDEX IF NOT EXISTS idx_evaluation_entry_entry_id
ON evaluation_entry(entry_id);

"""

In [ ]:
#-------------------------DATABASE CREATION----------------------#

def init_db(path:  str) -> sqlite3.Connection:
  conn = sqlite3.connect(path)
  conn.execute("PRAGMA foreign_keys = ON;")   # needed for ON DELETE CASCADE
  conn.execute("PRAGMA journal_mode = WAL;")  # safer/faster writes
  conn.executescript(EVALUATION_SCHEMA)
  conn.commit()
  return conn

In [ ]:
SUCCESS_THRESHOLD = 70
def summarize(eval_frame: pd.DataFrame):
    evaluated = eval_frame.dropna(subset=["judge_score_100"])
    if evaluated.empty:
        print("No judge results found. Run the evaluator first.")
        return
    print(f"Evaluated records: {len(evaluated)}")
    print(f"Mean Embedding Similarity (Cosine): {evaluated['embedding_similarity'].mean():.1f}/100")
    print(f"Mean Correctness Score: {evaluated['correctness_score'].mean()*20:.1f} %")
    print(f"Mean Coverage Score: {evaluated['coverage_score'].mean()*20:.1f} %")
    print(f"Mean Consistency Score: {evaluated['Consistency_score'].mean()*20:.1f} %")
    print(f"Mean Groundedness Score: {evaluated['groundedness_score'].mean()*20:.1f} %")
    print(f"Mean Relevance Score: {evaluated['relevance_score'].mean()*20:.1f} %")
    print(f"Mean Judge Score: {evaluated['judge_score_100'].mean():.1f}%")
    print(f"Success Rate (Score >= {SUCCESS_THRESHOLD}): {evaluated['success']}")